In [2]:
import xarray_behave as xb
import matplotlib.pyplot as plt
import scipy
import numpy as np
import datetime
import pandas as pd

from tools import parse_log_file

plt.style.use('ncb.mplstyle')

In [11]:
datename='localhost-20260402_130508'
root='/Volumes/agauneu/#Data/playback'
suffix = '_2' # or ''
filename = f"{datename}{suffix}"

ds = xb.assemble(datename=datename, root=root, target_sampling_rate=1_000, resample_video_data=True, filepath_poses=f"{root}/res/{datename}/{filename}_sleap.h5", filepath_video=f"{root}/res/{datename}/{filename}_sleap.h5")
nb_flies = len(ds.flies)
ds

   Failed loading movie params - no ss_movie


<xarray.Dataset> Size: 3GB
Dimensions:              (sampletime: 17995000, channels: 1, time: 3597984,
                          event_types: 0, event_time: 3, index: 0,
                          poseparts: 2, coords: 2, flies: 10)
Coordinates:
  * sampletime           (sampletime) float64 144MB 0.0 0.0002 ... 3.599e+03
  * time                 (time) float64 29MB 0.0 0.0009999 ... 3.598e+03
    nearest_frame        (time) int64 29MB 304 305 305 ... 216173 216173 216173
  * event_types          (event_types) float64 0B 
    event_categories     (event_types) float64 0B 
  * event_time           (event_time) <U13 156B 'start_seconds' ... 'channels'
  * poseparts            (poseparts) <U4 32B 'head' 'tail'
  * coords               (coords) <U1 8B 'y' 'x'
Dimensions without coordinates: channels, index, flies
Data variables:
    song_raw             (sampletime, channels) float64 144MB 4.02e-05 ... -0...
    song_events          (time, event_types) int16 0B 
    event_times          (index, event_time) float64 0B 
    event_names          (index) <U128 0B 
    pose_positions       (time, flies, poseparts, coords) float64 1GB 0.0 ......
    pose_positions_allo  (time, flies, poseparts, coords) float64 1GB 629.6 ....
Attributes:
    video_filename:           /Volumes/agauneu/#Data/playback/dat/localhost-2...
    datename:                 localhost-20260402_130508
    root:                     /Volumes/agauneu/#Data/playback
    dat_path:                 dat
    res_path:                 res
    sampling_rate_Hz:         5000.0
    target_sampling_rate_Hz:  1000
    ref_time:                 1775127920.2991

### Parse logs

In [ ]:
logs = parse_log_file(f"{root}/dat/{datename}/{datename}_daq.log")
stim_names = [s[0] for s in logs['stimFileName']]
logs.head()

,timestamp,rig,cnt,stimFileName,silencePre,silencePost,delayPost,intensity,freq,MODE
0,2026-04-02 13:05:20.264,DAQ@PC-845393,1,[santune_ipi70_pulseTrain_PDUR30ms_PCAR170hz_P...,[30000],[30000],[0],[6.0],[200.0],[nan]
1,2026-04-02 13:06:22.855,DAQ@PC-845393,2,[santune_ipi38_pulseTrain_PDUR30ms_PCAR170hz_P...,[30000],[30000],[0],[6.0],[200.0],[nan]
2,2026-04-02 13:07:26.857,DAQ@PC-845393,3,[santune_ipi130_pulseTrain_PDUR30ms_PCAR170hz_...,[30000],[30000],[0],[6.0],[200.0],[nan]
3,2026-04-02 13:08:30.860,DAQ@PC-845393,4,[santune_ipi70_pulseTrain_PDUR30ms_PCAR170hz_P...,[30000],[30000],[0],[6.0],[200.0],[nan]
4,2026-04-02 13:09:34.862,DAQ@PC-845393,5,[santune_ipi38_pulseTrain_PDUR30ms_PCAR170hz_P...,[30000],[30000],[0],[6.0],[200.0],[nan]


### Compute speed

In [5]:
pos = np.nanmean(ds.pose_positions_allo, axis=-2)
speed_raw = np.linalg.norm(xb.metrics.velocity(pos), axis=-1)
speed = scipy.signal.medfilt(speed_raw, 5)


/var/folders/zr/6ql4dzjx0tq8mpzht_2dwh480000gn/T/ipykernel_3142/3299271039.py:1: RuntimeWarning: Mean of empty slice
  pos = np.nanmean(ds.pose_positions_allo, axis=-2)


### Detect stimulus on and offsets

In [6]:
fs = ds.attrs['target_sampling_rate_Hz']
win_len = int(0.05 * fs)
window = scipy.signal.windows.gaussian(win_len, win_len // 8)
window /= np.sum(window)

env = np.sqrt(np.convolve(ds.song_raw[:, 0]**2, window, mode='full'))
env = env[win_len//2:win_len//2 + ds.song_raw.shape[0]]

In [7]:
thres = 1
stim = scipy.ndimage.binary_closing(env>thres, np.ones(1001))
stim_diff = np.diff(stim, prepend=0).astype(float)
stim_onsets = np.where(stim_diff==1)[0]
stim_offsets = np.where(stim_diff==-1)[0]

onset_times = ds.sampletime[stim_onsets]
offset_times = ds.sampletime[stim_offsets]


In [8]:
fs = ds.attrs['target_sampling_rate_Hz']
prefix = int(5 * fs)
duration = int(10 * fs)

all_traces = np.zeros((len(stim_onsets), prefix + duration, nb_flies))
for cnt, onset_time in enumerate(onset_times):
    onset_idx = np.argmax(ds.time.data[:, np.newaxis]>=onset_time.data)
    all_traces[cnt, :, :] = speed[onset_idx - prefix:onset_idx + duration, :]

all_traces = np.array(all_traces)

In [ ]:
time = np.arange(-prefix, duration) / fs
time

array([-5.   , -4.999, -4.998, ...,  9.997,  9.998,  9.999],
      shape=(15000,))

In [38]:
logs = logs.iloc[:len(onset_times)]
logs['onset_times'] = np.asarray(onset_times)
logs['offset_times'] = np.asarray(offset_times)

logs

,timestamp,rig,cnt,stimFileName,silencePre,silencePost,delayPost,intensity,freq,MODE,onset_times,offset_times
0,2026-04-02 13:05:20.264,DAQ@PC-845393,1,[santune_ipi70_pulseTrain_PDUR30ms_PCAR170hz_P...,[30000],[30000],[0],[6.0],[200.0],[nan],30.974386,34.909551
1,2026-04-02 13:06:22.855,DAQ@PC-845393,2,[santune_ipi38_pulseTrain_PDUR30ms_PCAR170hz_P...,[30000],[30000],[0],[6.0],[200.0],[nan],94.965852,98.933078
2,2026-04-02 13:07:26.857,DAQ@PC-845393,3,[santune_ipi130_pulseTrain_PDUR30ms_PCAR170hz_...,[30000],[30000],[0],[6.0],[200.0],[nan],158.958287,162.743515
3,2026-04-02 13:08:30.860,DAQ@PC-845393,4,[santune_ipi70_pulseTrain_PDUR30ms_PCAR170hz_P...,[30000],[30000],[0],[6.0],[200.0],[nan],222.860700,226.796922
4,2026-04-02 13:09:34.862,DAQ@PC-845393,5,[santune_ipi38_pulseTrain_PDUR30ms_PCAR170hz_P...,[30000],[30000],[0],[6.0],[200.0],[nan],286.853649,290.820875
5,2026-04-02 13:10:38.864,DAQ@PC-845393,6,[santune_ipi130_pulseTrain_PDUR30ms_PCAR170hz_...,[30000],[30000],[0],[6.0],[200.0],[nan],350.846057,354.631278
6,2026-04-02 13:11:42.667,DAQ@PC-845393,7,[santune_ipi70_pulseTrain_PDUR30ms_PCAR170hz_P...,[30000],[30000],[0],[6.0],[200.0],[nan],414.748456,418.683681
7,2026-04-02 13:12:46.669,DAQ@PC-845393,8,[santune_ipi38_pulseTrain_PDUR30ms_PCAR170hz_P...,[30000],[30000],[0],[6.0],[200.0],[nan],478.740856,482.708081
8,2026-04-02 13:13:50.671,DAQ@PC-845393,9,[santune_ipi130_pulseTrain_PDUR30ms_PCAR170hz_...,[30000],[30000],[0],[6.0],[200.0],[nan],542.732825,546.518546
9,2026-04-02 13:14:54.674,DAQ@PC-845393,10,[santune_ipi70_pulseTrain_PDUR30ms_PCAR170hz_P...,[30000],[30000],[0],[6.0],[200.0],[nan],606.635198,610.570967


In [39]:
save_file_name = f'{root}/res/{datename}/{filename}_spd.npz'
np.savez(save_file_name, traces=all_traces, time=time, **{k: list(v) for k, v in logs.items()})